<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 3 - Ednalvo

## Import `500_redacoes_AIRiAL_refeita.xlsx` into a DataFrame

In [1]:
from pathlib import Path
import pandas as pd

xlsx_path = Path("corpus/00_fontes/500_redacoes_AIRiAL_refeita.xlsx")

df_university_entrance_compositions = pd.read_excel(xlsx_path)

## Rename columns and standardise the `grupo` column

In [2]:
df_university_entrance_compositions = df_university_entrance_compositions.drop(index=0).reset_index(drop=True)

df_university_entrance_compositions.columns = [
    "grupo",
    "inscricao",
    "adequacao_ao_tema",
    "adequacao_a_coletanea",
    "adequacao_ao_tipo_de_texto",
    "adequacao_a_norma_padrao",
    "coesao",
    "coerencia",
    "pontuacao_total",
]

df_university_entrance_compositions["grupo"] = (
    df_university_entrance_compositions["grupo"]
    .replace(
        {
            "250 maiores notas": "250_maiores_notas",
            "250 menores notas": "250_menores_notas",
        }
    )
)

## Add the `arquivo_original` column

In [3]:
from pathlib import Path

source_dir = Path(
    "../cl_st1_ph3_ednalvo/corpus/00_fontes/500_redacoes_airial_maiores_e_menores_notas"
)

filename_by_inscricao = {
    path.name[:5]: path.name
    for path in source_dir.glob("*.txt")
    if path.name[:5].isdigit()
}

arquivo_original = (
    df_university_entrance_compositions["inscricao"]
    .astype(str)
    .str.zfill(5)
    .map(filename_by_inscricao)
)

inscricao_position = df_university_entrance_compositions.columns.get_loc("inscricao")

if "arquivo_original" in df_university_entrance_compositions.columns:
    df_university_entrance_compositions = df_university_entrance_compositions.drop(
        columns="arquivo_original"
    )

df_university_entrance_compositions.insert(
    inscricao_position + 1,
    "arquivo_original",
    arquivo_original,
)

## Create the `anotacoes` column

Identify the compositions that have annotations.

In [4]:
from pathlib import Path
import re

source_dir = Path(
    "../cl_st1_ph3_ednalvo/corpus/00_fontes/500_redacoes_airial_maiores_e_menores_notas"
)

annotation_pattern = re.compile(r"\{[^}]*\}|\[[^\]]*\]|\([^)]*\)")
essay_id_pattern = re.compile(r"\(\d{4}-\d{3}\)")

def extract_annotations(filename):
    if pd.isna(filename):
        return pd.NA

    path = source_dir / filename

    if not path.exists():
        return pd.NA

    text = path.read_text(encoding="utf-8")
    annotations = []

    for match in annotation_pattern.findall(text):
        inner = match[1:-1]

        # Skip multiline spans and filename/header-like essay IDs, e.g. (0463-010)
        if "\n" in inner or "\r" in inner:
            continue
        if essay_id_pattern.fullmatch(match):
            continue

        annotations.append(match)

    return "; ".join(annotations) if annotations else pd.NA


anotacoes = df_university_entrance_compositions["arquivo_original"].apply(
    extract_annotations
)

arquivo_original_position = df_university_entrance_compositions.columns.get_loc(
    "arquivo_original"
)

if "anotacoes" in df_university_entrance_compositions.columns:
    df_university_entrance_compositions = df_university_entrance_compositions.drop(
        columns="anotacoes"
    )

df_university_entrance_compositions.insert(
    arquivo_original_position + 1,
    "anotacoes",
    anotacoes,
)

In [7]:
valores_unicos_anotacoes = {
    anotacao.strip()
    for anotacoes_celula in df_university_entrance_compositions["anotacoes"].dropna()
    for anotacao in str(anotacoes_celula).split(";")
    if anotacao.strip()
}

valores_unicos_anotacoes = sorted(valores_unicos_anotacoes)

valores_unicos_anotacoes

['(...)',
 '(0429/001)',
 '(0463/002)',
 '(2015)',
 '(Ar)',
 '(CET)',
 "(CFC's)",
 '(CO₂)',
 '(Fogo, Ar, Água e Terra)',
 '(Fogo, Terra, Ar e Água)',
 '(Globo)',
 '(Gás Carbônico)',
 '(IBGE)',
 '(INPE)',
 '(Instituto Brasileiro de Geografia e Estatística)',
 '(MG)',
 '(Minas Gerais)',
 '(Minas Gerais, Brasil)',
 '(Naufrágio)',
 '(O rompimento das duas barragens de rejeitos minerais na cidade de Mariana, Minas Gerais)',
 '(O3)',
 '(ONG’s)',
 '(Organização não Governamental)',
 '(PA)',
 '(PAC)',
 '(PIB)',
 '(PMDB)',
 '(PSDB)',
 '(Pará)',
 '(Pausa para a publicidade: “Seguros + Deus: Sua salvação custa apenas uma oração”)',
 '(Preservação redobrada com a conscientização do homem com a natureza)',
 '(RGS)',
 '(TV, rádio, jornais)',
 '(Terra, água, fogo e ar)',
 '(UFMT)',
 '(USP)',
 '(a Terra é um organismo vivo que reage às agressões humanas)',
 '(a casa, os filhos e a essência)',
 '(a partir de leis)',
 '(a sensata)',
 '(a sensível)',
 '(agropecuária)',
 '(ar)',
 '(ar, fogo, água, terra)'

## Reorganise the files into folders based on the `grupo` column

In [5]:
from pathlib import Path
import shutil

source_dir = Path(
    "../cl_st1_ph3_ednalvo/corpus/00_fontes/500_redacoes_airial_maiores_e_menores_notas"
)

output_dirs = {
    "250_maiores_notas": Path("corpus/01_composicoes/maiores_notas"),
    "250_menores_notas": Path("corpus/01_composicoes/menores_notas"),
}

for output_dir in output_dirs.values():
    output_dir.mkdir(parents=True, exist_ok=True)

caminhos_em_composicoes = []

for _, row in df_university_entrance_compositions.iterrows():
    source_file = source_dir / row["arquivo_original"]

    if not source_file.exists():
        print(f"Source file not found: {source_file}")
        caminhos_em_composicoes.append(pd.NA)
        continue

    target_dir = output_dirs.get(row["grupo"])

    if target_dir is None:
        print(f"Unknown grupo: {row['grupo']}")
        caminhos_em_composicoes.append(pd.NA)
        continue

    target_file = target_dir / f"{str(row['inscricao']).zfill(5)}.txt"

    shutil.copy2(source_file, target_file)
    caminhos_em_composicoes.append(str(target_file))

if "caminho_em_composicoes" in df_university_entrance_compositions.columns:
    df_university_entrance_compositions = df_university_entrance_compositions.drop(
        columns="caminho_em_composicoes"
    )

arquivo_original_position = df_university_entrance_compositions.columns.get_loc("arquivo_original")

df_university_entrance_compositions.insert(
    arquivo_original_position + 1,
    "caminho_em_composicoes",
    caminhos_em_composicoes,
)

df_university_entrance_compositions

,grupo,inscricao,arquivo_original,caminho_em_composicoes,anotacoes,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,coerencia,pontuacao_total
0,250_maiores_notas,36237,36237 (0519-029).txt,corpus/01_composicoes/maiores_notas/36237.txt,{FO: inc findas},4.5,5,5,5,5,5,29.5
1,250_maiores_notas,46656,46656 (0414-009).txt,corpus/01_composicoes/maiores_notas/46656.txt,NaN,5,5,4,4.5,5,5,28.5
2,250_maiores_notas,36560,36560 (0502-031).txt,corpus/01_composicoes/maiores_notas/36560.txt,{FO: poluição}; {FO: más},4.5,4.5,4.5,5,5,4.5,28
3,250_maiores_notas,36923,36923 (0416-024).txt,corpus/01_composicoes/maiores_notas/36923.txt,NaN,5,4,4.5,5,5,4.5,28
4,250_maiores_notas,45355,45355 (0477-030).txt,corpus/01_composicoes/maiores_notas/45355.txt,"(terra, fogo, ar e água); (terra, fogo, ar e á...",4.5,4.5,4.5,5,5,4.5,28
...,...,...,...,...,...,...,...,...,...,...,...,...
495,250_menores_notas,34613,34613 (0472-031).txt,corpus/01_composicoes/menores_notas/34613.txt,{FO: horriveis}; {FO: hidreletricas}; {FO: equ...,1.5,1.5,2,1,3,2.5,11.5
496,250_menores_notas,34971,34971 (0492-012).txt,corpus/01_composicoes/menores_notas/34971.txt,{FO: frequênte}; {FO: catastrofis},1.5,1.5,2.5,1.5,2.5,2,11.5
497,250_menores_notas,35072,35072 (0496-007).txt,corpus/01_composicoes/menores_notas/35072.txt,NaN,1,1,2,3,3,1.5,11.5
498,250_menores_notas,35136,35136 (0505-029).txt,corpus/01_composicoes/menores_notas/35136.txt,NaN,2,2.5,2.5,0.5,1.5,2.5,11.5


## Exporting `df_university_entrance_compositions` as files

In [6]:
from pathlib import Path

export_dir = Path("corpus/00_fontes")
export_dir.mkdir(parents=True, exist_ok=True)

export_base_path = export_dir / "composicoes_de_admissao_universitaria"

df_university_entrance_compositions.to_json(
    export_base_path.with_suffix(".ndjson"),
    orient="records",
    lines=True,
    force_ascii=False,
)

df_university_entrance_compositions.to_csv(
    export_base_path.with_suffix(".tsv"),
    sep="	",
    index=False,
    encoding="utf-8",
)

df_university_entrance_compositions.to_excel(
    export_base_path.with_suffix(".xlsx"),
    index=False,
)

[
    export_base_path.with_suffix(".ndjson"),
    export_base_path.with_suffix(".tsv"),
    export_base_path.with_suffix(".xlsx"),
]

[PosixPath('corpus/00_fontes/composicoes_de_admissao_universitaria.ndjson'),
 PosixPath('corpus/00_fontes/composicoes_de_admissao_universitaria.tsv'),
 PosixPath('corpus/00_fontes/composicoes_de_admissao_universitaria.xlsx')]